# Part 3: Repeatable Evaluations with an Eval Harness  

After experimenting informally, we need a **systematic way to measure performance**. This part introduces an **evaluation harness**—a structured script for running batch tests across multiple inputs.  

We’ll cover:  
- **Defining evaluation criteria** based on observed patterns.  
- **Running batch tests** using a dataset of team names.  
- **Expanding evaluation** to real-world data, like news articles.  

By the end, you’ll have a **repeatable evaluation process**, ensuring that our AI system’s performance is measurable and consistent across different inputs.  


**Original workshop recording:** The original workshop used Gemma 2B; this notebook has been updated to Gemma 4. [Watch this section on YouTube.](https://www.youtube.com/live/9zM_93mYdu8?t=4104)

In [1]:
from ollama import chat
from ollama import ChatResponse

model = 'gemma4:latest'

def single_turn(prompt):
    response: ChatResponse = chat(model=model, messages=[
      {
        'role': 'user',
        'content': prompt,
      },
    ])
    return response['message']['content']

prompt = "Say hello to the class"
single_turn(prompt)

"Hello, class! 👋\n\nIt's wonderful to meet you all! I'm here and ready to help you learn, explore, and answer any questions you have.\n\nI'm looking forward to a great session with everyone! Let's get started! 😊"

## Your First Evals

Here are the list of AFL and NFL team names.
Check if the model gets things right.
This means you'll need to create a scoring and aggregation function.


In [2]:
afl_clubs = [
    "Adelaide Crows",
    "Brisbane Lions",
    "Carlton Blues",
    "Collingwood Magpies",
    "Essendon Bombers",
    "Fremantle Dockers",
    "Geelong Cats",
    "Gold Coast Suns",
    "Greater Western Sydney (GWS) Giants",
    "Hawthorn Hawks",
    "Melbourne Demons",
    "North Melbourne Kangaroos",
    "Port Adelaide Power",
    "Richmond Tigers",
    "St Kilda Saints",
    "Sydney Swans",
    "West Coast Eagles",
    "Western Bulldogs"
]

nfl_teams = [
    "Arizona Cardinals",
    "Atlanta Falcons",
    "Baltimore Ravens",
    "Buffalo Bills",
    "Carolina Panthers",
    "Chicago Bears",
    "Cincinnati Bengals",
    "Cleveland Browns",
    "Dallas Cowboys",
    "Denver Broncos",
    "Detroit Lions",
    "Green Bay Packers",
    "Houston Texans",
    "Indianapolis Colts",
    "Jacksonville Jaguars",
    "Kansas City Chiefs",
    "Las Vegas Raiders",
    "Los Angeles Chargers",
    "Los Angeles Rams",
    "Miami Dolphins",
    "Minnesota Vikings",
    "New England Patriots",
    "New Orleans Saints",
    "New York Giants",
    "New York Jets",
    "Philadelphia Eagles",
    "Pittsburgh Steelers",
    "San Francisco 49ers",
    "Seattle Seahawks",
    "Tampa Bay Buccaneers",
    "Tennessee Titans",
    "Washington Commanders"
]

In [3]:
import numpy as np
eval_map = {"australian": afl_clubs, "american": nfl_teams}

# Remove key let students code themselves
score = []
for nationality, teams in eval_map.items():
    for team in teams:
        prompt = f"Output if this is an australian or american team, only print australian or american no other output: {team}"
        response = single_turn(prompt).strip()
        score.append(response.lower() == nationality)
        print(f"{team}: {response}")

print(np.array(score).mean())

Adelaide Crows: australian
Brisbane Lions: australian
Carlton Blues: australian
Collingwood Magpies: australian
Essendon Bombers: australian
Fremantle Dockers: australian
Geelong Cats: australian
Gold Coast Suns: australian
Greater Western Sydney (GWS) Giants: australian
Hawthorn Hawks: australian
Melbourne Demons: australian
North Melbourne Kangaroos: australian
Port Adelaide Power: australian
Richmond Tigers: australian
St Kilda Saints: australian
Sydney Swans: australian
West Coast Eagles: australian
Western Bulldogs: australian
Arizona Cardinals: american
Atlanta Falcons: american
Baltimore Ravens: american
Buffalo Bills: american
Carolina Panthers: american
Chicago Bears: american
Cincinnati Bengals: american
Cleveland Browns: american
Dallas Cowboys: american
Denver Broncos: american
Detroit Lions: american
Green Bay Packers: american
Houston Texans: american
Indianapolis Colts: american
Jacksonville Jaguars: american
Kansas City Chiefs: american
Las Vegas Raiders: american
Los A

## From Notebook to Script: Making Evaluation Repeatable

Now that we’ve run our evaluation in the notebook, we want to make this process **repeatable and scriptable**.  

Executing the following cell will **generate `traces.csv`**, storing the model's predictions alongside the ground truth.  

Once the file is created, run the evaluation harness script to **compute accuracy programmatically**:  

```
python eval_harness.py data/traces.csv --output data/scored_results.csv
```

In [9]:
import csv

eval_map = {"australian": afl_clubs, "american": nfl_teams}

# Open CSV file for writing
csv_filename = "data/traces.csv"
with open(csv_filename, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["team_name", "ground_truth", "model_output"])

    for nationality, teams in eval_map.items():
        for team in teams:
            prompt = f"Output if this is an australian or american team, only print australian or american no other output {team}"
            response = single_turn(prompt).strip().lower()
            
            # Save to CSV
            writer.writerow([team, nationality, response])
            
            print(f"{team}: {response}")

print(f"📁 Saved results to {csv_filename}")

Adelaide Crows: australian
Brisbane Lions: australian
Carlton Blues: australian
Collingwood Magpies: australian
Essendon Bombers: australian
Fremantle Dockers: australian
Geelong Cats: australian
Gold Coast Suns: australian
Greater Western Sydney (GWS) Giants: australian
Hawthorn Hawks: australian
Melbourne Demons: australian
North Melbourne Kangaroos: australian
Port Adelaide Power: australian
Richmond Tigers: australian
St Kilda Saints: australian
Sydney Swans: australian
West Coast Eagles: australian
Western Bulldogs: australian
Cardinals: american
Falcons: (i cannot accurately determine the origin of "falcons" without context.)
Ravens: american
Bills: american
Panthers: american
Bears: american
Bengals: american
Browns: american
Cowboys: american
Broncos: american
Lions: american
Packers: american
Texans: american
Colts: american
Jaguars: american
Chiefs: american
Raiders: american
Chargers: american
Rams: american
Dolphins: american
Vikings: american
Patriots: american
Saints: ame

## Making things harder. Taking just the team name
What happens if we don't provide the full context. What happens to our score then? Is this expected?

In [5]:
afl_names = ['Crows',
 'Lions',
 'Blues',
 'Magpies',
 'Bombers',
 'Dockers',
 'Cats',
 'Suns',
 'Giants',
 'Hawks',
 'Demons',
 'Kangaroos',
 'Power',
 'Tigers',
 'Saints',
 'Swans',
 'Eagles',
 'Bulldogs']

In [6]:
nfl_teams = ['Cardinals',
 'Falcons',
 'Ravens',
 'Bills',
 'Panthers',
 'Bears',
 'Bengals',
 'Browns',
 'Cowboys',
 'Broncos',
 'Lions',
 'Packers',
 'Texans',
 'Colts',
 'Jaguars',
 'Chiefs',
 'Raiders',
 'Chargers',
 'Rams',
 'Dolphins',
 'Vikings',
 'Patriots',
 'Saints',
 'Giants',
 'Jets',
 'Eagles',
 'Steelers',
 '49ers',
 'Seahawks',
 'Buccaneers',
 'Titans',
 'Commanders']

In [7]:
eval_map = {"american": nfl_teams, "australian": afl_names}

core = []

for nationality, teams in eval_map.items():
    for team in teams:
        team = team.rsplit(maxsplit =1)[-1]
        prompt = "Output if this is an australian or american team, only print australian or american no other output: " + f"{team}"
        response = single_turn(prompt).strip()
        score.append(response.lower() == nationality)
        print(f"{team}: {response}")
print(np.array(score).mean())

Cardinals: american
Falcons: american
Ravens: american
Bills: american
Panthers: american
Bears: american
Bengals: american
Browns: american
Cowboys: american
Broncos: australian
Lions: american
Packers: american
Texans: american
Colts: american
Jaguars: american
Chiefs: american
Raiders: american
Chargers: american
Rams: american
Dolphins: american
Vikings: american
Patriots: american
Saints: australian
Giants: american
Jets: american
Eagles: american
Steelers: american
49ers: american
Seahawks: american
Buccaneers: american
Titans: australian
Commanders: american
Crows: american
Lions: american
Blues: american
Magpies: australian
Bombers: i cannot determine the origin without more context
Dockers: american
Cats: australian
Suns: american
Giants: american
Hawks: american
Demons: australian
Kangaroos: australian
Power: 
Tigers: american
Saints: american
Swans: australian
Eagles: american
Bulldogs: australian
0.85


## News Articles
Let's now try this on full articles. We've provided two samples for you but you're going to need todo the bulk of the work here. That is

1. Construct a larger eval set
2. Store the expected result in a way that you can check the model response
3. Run each article and check the result
4. Aggregate the results and create a final metric

In [8]:
import glob
txt_files = glob.glob("text_sources/*.txt")

for txt_file in txt_files:
    with open(txt_file, 'r') as f:
        article_text = f.read()
        full_prompt = f"print if this article is about an australian or american team, no other output: {article_text}"

        response = single_turn(full_prompt).strip()
        print(response)

australian
American


##  Recap: What We Learned  

### Setting up repeatable eval infra
* Evaling by vibes is great start but
  * It's manual to rerun
  * Hard to compare as projects scale
  * Ambiguous to track as things change

### Building your own evalsets
* Evalsets are as critical as model and infra choice
* No one knows your use case better than you
* Building your own evalset will encourage deep thinking about your challenge